# Session 2 — AI Agents Workshop
Code companion, ordered to match the deck ("Martin and laura").


## 📍 Slide section 01 — How LLMs Work
*(Perceptron, pretraining, SFT, RLHF — no code, slides only)*


## ⚙️ Setup — run before anything else


In [40]:
# ------------------SETUP------------------------

!pip install -q groq
from groq import Groq
from google.colab import userdata
import time

API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=API_KEY)
import requests

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {API_KEY}"}
)
MODEL = ""

In [39]:
import requests
resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {API_KEY}"}
)
for m in resp.json()["data"]:
    print(m["id"])

qwen/qwen3.6-27b
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b
groq/compound-mini
qwen/qwen3.8-27b
allam-2-7b
openai/gpt-oss-120b
groq/compound
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b


In [41]:
# Defining LLM
def call_llm(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(model=MODEL, messages=messages)
    text = resp.choices[0].message.content
    if not text:
        raise ValueError("LLM returned an empty response.")
    return text.strip()




You are all set! Groq replied: Hello!


## 🔧 Slide section 03 — Tool Calls & Looping
Reshuffled per your notes: **tool-call definitions first**, then looping motivation together.

### 3a. "Let's give Atlas power" / "TOOL CALLS:" (definition slides)
This is literally what a tool is — a named Python function the agent can call.


In [42]:
# ── STEP 2: define tools ──────────────────────────────
def mock_search(query):
    facts = {"ceo of google": "Sundar Pichai", "capital of japan": "Tokyo", "ceo of marlowe dynamics": "Renata Ibarra"}
    return facts.get(query.lower().strip().rstrip("?"), "No result found.")


### 3b. "WHO CHOOSES? WHO EXECUTES?" — types of tool calls / choosing a tool
`which_tool()` is the **choosing** half — no execution yet. That split (choose vs. execute) is exactly what this slide is asking.


### 3c. "Why use more than one tool call?" — looping motivation
No new code here — reuse the **apple question** from section 02 as the live example: it needs *sequential* subtraction/multiplication steps, which is why one tool call isn't enough. Say it out loud, then move straight into the ReAct system prompt below.


## 🔁 Slide section — Reason + Act (ReAct Model)
This is the missing loop-mechanism slide (Thought → Action → Observation). `SYSTEM_PROMPT` is the loop's vocabulary; `agent()` is the loop itself.


In [59]:
SYSTEM_PROMPT = """You are Atlas, a helpful agent that can use tools to reach a goal.
Available tools:
- calculator: does maths, e.g. "12 * 8"
- search: looks up simple facts, e.g. "capital of japan"

On every turn, reply with EXACTLY ONE line, in ONE of these two formats:
ACTION: tool_name: input
ANSWER: your final answer

Use ACTION when you still need information or a calculation.
Use ANSWER as soon as you can fully answer the goal.
Never write anything outside of those two formats."""

### The loop itself — map this line-by-line to the Thought / Action / Observation diagram


In [64]:
def agent(goal, max_steps=4):
    history = f"Goal: {goal}"
    for step in range(max_steps):
        print(f"\n--- Step {step + 1} ---")
        try:
            response = call_llm(history, system=SYSTEM_PROMPT).strip()
        except Exception as e:
            return f"LLM error: {repr(e)}"
        # print("Atlas says:", repr(response))
        time.sleep(2)

        if response.startswith("ANSWER:"):
            return response.split("ANSWER:", 1)[1].strip()

        elif response.startswith("ACTION:"):
            try:
                # Format: ACTION: tool_name: input
                action_body = response.split("ACTION:", 1)[1].strip()
                tool_name, tool_input = action_body.split(":", 1)
                tool_name = tool_name.strip()
                tool_input = tool_input.strip()
            except Exception:
                print("Could not parse action.")
                history += f"\n{response}\nOBSERVATION: Invalid format. Use exactly:\nACTION: tool_name: input"
                continue

            if tool_name in tools:
                try:
                    result = tools[tool_name](tool_input)
                except Exception as e:
                    result = f"Tool error: {repr(e)}"
                print(f"Tool '{tool_name}' returned:", result)
                history += f"\n{response}\nOBSERVATION: {result}"
            else:
                print("Unknown tool:", tool_name)
                history += f"\n{response}\nOBSERVATION: Unknown tool '{tool_name}'."
        else:
            print("Invalid response format.")
            history += f"\n{response}\nOBSERVATION: Invalid response format. Respond using either ACTION or ANSWER."

    return "Agent ran out of steps without a final answer."